# Nested Relation Dataset Maker

This notebook generates nested relations, their text representation and then asks an LLM to generate a sentence that describes that nested relation.

First, let's open up a schema that defines the possible relations:

In [2]:
import yaml

with open("relation_schema.yml") as f:
    relation_schema = yaml.safe_load(f)

And also load a term list of genes/chemicals/etc that we'll use in our made-up relations/sentences:

In [3]:
import json
with open('terms.json') as f:
    terms = json.load(f)
terms['biomolecule'] = terms['chemical'] + terms['gene']
terms['gene_protein'] = terms['gene']

## Generate a single relation

Let's work through making a single nested relation (before we mass-produce them).

In [4]:
import random
random.seed(42)

We're going to make a function that picks a relation, and then recursively fills in its arguments (with either a sub-relation or an entity).

In [5]:
def generate_nested_relation(choices=list(relation_schema.keys())):
    event = random.choice(choices)
    if event not in relation_schema:
        assert event in terms
        selected_entity = random.choice(terms[event])
        return {'type':'entity', 'entity':selected_entity}
    
    definition = relation_schema[event]
    arg_values = {}
    for arg in definition['args']:
        arg_types = definition['arg_types'][arg] if 'arg_types' in definition else [arg]
        arg_values[arg] = generate_nested_relation(arg_types)

    return {'type':'relation', 'label':event, 'arguments':arg_values}

relation = generate_nested_relation(['effect'])
relation

{'type': 'relation',
 'label': 'effect',
 'arguments': {'cause': {'type': 'relation',
   'label': 'effect',
   'arguments': {'cause': {'type': 'relation',
     'label': 'increase',
     'arguments': {'variable': {'type': 'entity',
       'entity': 'complex cortical dysplasia with other brain malformations 15'}}},
    'theme': {'type': 'relation',
     'label': 'expression',
     'arguments': {'gene_protein': {'type': 'entity', 'entity': 'CD40LG'}}}}},
  'theme': {'type': 'relation',
   'label': 'methylation',
   'arguments': {'gene_protein': {'type': 'entity', 'entity': 'CLU'}}}}}

Now we need to take that nest of dictionaries and turn it into a single string representation:

In [12]:
def make_relation_text(relation):
    if relation['type'] == 'relation':
        label = relation['label']
        delimited_arguments = ", ".join( f"{arg}={make_relation_text(value)}" for arg,value in relation['arguments'].items() )
        return f"{label}({delimited_arguments})"
    else:
        return f"[QUOTE]{relation['entity']}[QUOTE]"

relation_text = make_relation_text(relation)
relation_text

'effect(cause=effect(cause=increase(variable=[QUOTE]complex cortical dysplasia with other brain malformations 15[QUOTE]), theme=expression(gene_protein=[QUOTE]CD40LG[QUOTE])), theme=methylation(gene_protein=[QUOTE]CLU[QUOTE]))'

We'll also want to know just the entities that are mentioned.

In [7]:
def get_entities(relation):
    if relation['type'] == 'relation':
        entities = []
        for arg,value in relation['arguments'].items():
            entities += get_entities(value)
        return entities
    else:
        return [relation['entity']]

get_entities(relation)

['complex cortical dysplasia with other brain malformations 15',
 'CD40LG',
 'CLU']

And lastly, we'll need to ask an LLM to give us a sentence that contains the relation:

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import os

class Sentence(BaseModel):
    text: str

BASE_URL = "http://api.llm.apps.os.dcs.gla.ac.uk/v1"  # outside UofG network: "http://api.terrier.org/v1"
API_KEY = os.environ['IDA_LLM_API_KEY']
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def generate_sentence(relation_text):
    response = client.responses.parse(
        model="gpt-oss-120b",
        input=[
            {
                "role": "system",
                "content": "You are a biomedical science expert. The user will give you a nested set of relations. You will write a complex sentence (that could be in biomedical research article) that contains the relations. It should only contain the described relation(s) and no other. Make it long and complex.",
            },
            {"role": "user", "content": relation_text},
        ],
        text_format=Sentence,
    )
    
    sentence = response.output_parsed
    return sentence.text
    
generate_sentence(relation_text)

## Mass-produce them

Now let's make lots of them. For now, we'll create all relations with a base `effect` relation.

In [ ]:
import json
from tqdm.auto import tqdm

outputs = []
for i in tqdm(range(100000)):
    relation = generate_nested_relation(['effect'])
    relation_text = make_relation_text(relation)
    entities = get_entities(relation)
    sentence = generate_sentence(relation_text)

    output = { 'relation_text':relation_text, 'sentence':sentence, 'entities':entities, 'relation':relation }
    outputs.append(output)

    if (i%10) == 0:
        with open('nested_relations_dataset.json','w') as f:
            json.dump(outputs,f,indent=2)

# Re-working dataset for special QUOTE tokens

In [13]:
import json
import sys
from lark import Lark, exceptions

def regenerate_dataset(entries):
    """
    Overwrites `relation_text` for every entry using the updated renderer.
    Returns (updated_entries, problems) where problems is a list of
    (index, old_text, new_text, error) for anything that failed to
    round-trip parse after regeneration.
    """
    problems = []
    for i, entry in enumerate(entries):
        new_text = make_relation_text(entry['relation'])
        _, err = parse_tree(new_text)
        if err is not None:
            problems.append((i, entry.get('relation_text'), new_text, err))
        entry['relation_text'] = new_text
    return entries, problems

 
GRAMMAR = r"""
    start: predicate
    predicate: NAME "(" [arg ("," arg)*] ")"
    arg: NAME "=" value
    value: predicate
         | STRING
    NAME: /[a-zA-Z_][a-zA-Z0-9_]*/
    STRING: /(?s)\[QUOTE\].*?\[QUOTE\]/
    %import common.WS
    %ignore WS
"""
parser = Lark(GRAMMAR, start="start", parser="earley")
 
 
def parse_tree(text):
    try:
        return parser.parse(text), None
    except exceptions.UnexpectedCharacters as e:
        return None, f"unexpected_char at pos {e.pos_in_stream}: {e}"
    except exceptions.UnexpectedEOF:
        return None, "truncated_input"
    except exceptions.UnexpectedToken as e:
        return None, f"unexpected_token at pos {e.pos_in_stream}: {e}"
    except exceptions.LarkError as e:
        return None, f"parse_error: {e}"

In [15]:
in_path = "nested_relations_dataset.json"
out_path = "nested_relations_dataset_v2.json"

with open(in_path, "r", encoding="utf-8") as f:
    data = json.load(f)
 
    updated, problems = regenerate_dataset(data)
 
    if problems:
        print(f"WARNING: {len(problems)} / {len(data)} entries failed to round-trip parse:")
        for idx, old, new, err in problems[:20]:  # cap printed examples
            print(f"  [{idx}] old={old!r}")
            print(f"        new={new!r}")
            print(f"        error={err}")
        if len(problems) > 20:
            print(f"  ... and {len(problems) - 20} more")
        print("\nNot writing output file until these are resolved.")
        sys.exit(1)
 
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(updated, f, indent=2, ensure_ascii=False)
 
    print(f"Regenerated relation_text for {len(updated)} entries -> {out_path}")

Regenerated relation_text for 78761 entries -> nested_relations_dataset_v2.json
